In [14]:
from itertools import combinations, product

coldata = {'a': [1, 2], 'b': [3, 4], 'c': [5, 6]}

all_combinations = []

for group_size in range(1, len(coldata) + 1):
    for combination_keys in combinations(coldata.keys(), group_size):
        values_combinations = []
        for key in combination_keys:
            values_combinations.append(coldata[key])
        for combo in product(*values_combinations):
            combined_dict = {key: value for key, value in zip(combination_keys, combo)}
            all_combinations.append(combined_dict)

print(all_combinations)


[{'a': 1}, {'a': 2}, {'b': 3}, {'b': 4}, {'c': 5}, {'c': 6}, {'a': 1, 'b': 3}, {'a': 1, 'b': 4}, {'a': 2, 'b': 3}, {'a': 2, 'b': 4}, {'a': 1, 'c': 5}, {'a': 1, 'c': 6}, {'a': 2, 'c': 5}, {'a': 2, 'c': 6}, {'b': 3, 'c': 5}, {'b': 3, 'c': 6}, {'b': 4, 'c': 5}, {'b': 4, 'c': 6}, {'a': 1, 'b': 3, 'c': 5}, {'a': 1, 'b': 3, 'c': 6}, {'a': 1, 'b': 4, 'c': 5}, {'a': 1, 'b': 4, 'c': 6}, {'a': 2, 'b': 3, 'c': 5}, {'a': 2, 'b': 3, 'c': 6}, {'a': 2, 'b': 4, 'c': 5}, {'a': 2, 'b': 4, 'c': 6}]


In [14]:
class MyClass:
    def __init__(self):
        # 初始化类的属性 a1, a2, ... a100
        for i in range(1, 101):
            setattr(self, f'a{i}', i)

    def sum_attributes(self):
        total = 0
        for i in range(1, 101):
            total += getattr(self, f'a{i}')
        return total

# 创建类的实例
my_instance = MyClass()

# 调用 sum_attributes 方法求和
result = my_instance.sum_attributes()

print("总和:", result)
print(my_instance.a1)


列名: ['A', 'B', 'C']


In [4]:
import networkx as nx
from networkx.readwrite import json_graph
import pickle
# import numpy as np

# settings = {}
# with open("./settings.json", 'r') as f:
#     settings = json.load(f)
    
fn = "./NewSqls/SalariesFrance/workload/7.pkl"

with open(fn, "rb") as f:
    query = pickle.load(f)

query

{'sql': "SELECT * FROM   salariesfrance_13  WHERE  ((salariesfrance_13.reg_lib = 'BOURGOGNE-FRANCHE-COMTÉ') AND (salariesfrance_13.ze2010_lib = 'COSNE - CLAMECY'));",
 'join_graph': {'directed': False,
  'multigraph': False,
  'graph': [],
  'nodes': [{'predicates': [], 'id': 'salariesfrance_13'}],
  'adjacency': [[]]},
 'subset_graph': {'directed': True,
  'multigraph': False,
  'graph': [],
  'nodes': [{'cardinality': {'expected': 134376,
     'actual': 134376,
     'total': 134376},
    'exec_time': {'actual': 0.04474306106567383},
    'id': ('salariesfrance_13',)}],
  'adjacency': [[]]}}

In [5]:
import os
import shutil

def transform_quotes_to_lowercase(line):
    result = []
    in_quotes = False
    current_word = ""
    
    for char in line:
        if char == '"':
            if in_quotes:
                result.append(current_word.lower().replace(" ", ""))
                current_word = ""
            in_quotes = not in_quotes
        else:
            if in_quotes:
                current_word += char
            else:
                result.append(char)
    
    return ''.join(result)

print("========================================runnning bi_0_0_sql_process.py========================================")
# folder_path = "./bibench_processed"
# folder_path = "./NewSqls"
folder_path = "./NewSqls"

# 遍历文件夹
dir_list = os.listdir(folder_path)

notEqual = 0

for dir_name in dir_list:
    if ".csv" in dir_name:
        continue
    
    # 输入文件夹和输出文件夹的路径
    input_folder = os.path.join(folder_path, dir_name, "queries")
    # output_folder = input_folder.replace("queries", "queries_processed")

    # # 确保输出文件夹存在，如果不存在则创建
    # if not os.path.exists(output_folder):
    #     os.makedirs(output_folder)
    # else:
    #     shutil.rmtree(output_folder) # 删除文件夹
    #     os.makedirs(output_folder)  # 重新创建文件夹

    # 列出输入文件夹下的所有文件
    files = os.listdir(input_folder)

    # 遍历文件列表
    for filename in files:
        
        # 检查文件是否为sql文件（忽略大小写）
        if filename.lower().endswith(".sql"):
            # 构造输入文件的完整路径
            input_path = os.path.join(input_folder, filename)
            
            # 构造输出文件的完整路径，保持原文件名但转为小写
            # output_filename = filename.lower()
            # output_path = os.path.join(output_folder, output_filename)
            
            # 读取输入文件内容并转为小写，然后写入输出文件
            with open(input_path, "r") as input_file:
                sql_content = input_file.read()
                
                # 滤去(没有where)或(and和or都有的)
                if("WHERE" not in sql_content) or (" AND " in sql_content and " OR " in sql_content):
                    # print(input_path)
                    # print(sql_content)
                    continue
                
                # print(sql_content)
                sql_content = transform_quotes_to_lowercase(sql_content)
                sql_content = sql_content.replace("double", "double precision")
                
                sql_content = sql_content.replace("((", "(")
                sql_content = sql_content.replace("))", ")")
                
                if ">" in sql_content or "<" in sql_content:
                    notEqual += 1
                    print(input_path,"*******", sql_content)
                # print(sql_content)
                
                # with open(output_path, "w") as output_file:
                #     output_file.write(sql_content)
print(notEqual)

========================================runnning bi_0_0_sql_process.py========================================
./NewSqls/Arade/queries/2.sql ******* SELECT * FROM   arade_1  WHERE  (arade_1.f3 >= '2014-10-17') AND (arade_1.f3 <= '2015-10-16');
./NewSqls/Generico/queries/1.sql ******* SELECT * FROM  generico_1  WHERE  (generico_1.anunciante IN ('BANTRAB/TODOTICKET', 'TODOTICKET', 'TODOTICKET.COM') AND generico_1.fecha >= '2010-01-01' and generico_1.fecha <= '2015-12-31');
./NewSqls/Generico/queries/13.sql ******* SELECT * FROM  generico_5  WHERE  (generico_5.anunciante IN ('BANTRAB/TODOTICKET', 'TODOTICKET', 'TODOTICKET.COM') AND generico_5.fecha >= '2010-01-01' and generico_5.fecha <= '2015-12-31' AND (generico_5.medio = 'TELEVISION NACIONAL');
./NewSqls/Generico/queries/15.sql ******* SELECT * FROM  generico_5  WHERE  (generico_5.anunciante IN ('BANTRAB/TODOTICKET', 'TODOTICKET', 'TODOTICKET.COM') AND (generico_5.fecha >= '2014-01-01' and generico_5.fecha <= '2015-12-31') AND (generic